# Day 5: Structured Output

LangChain in 10 Days · KGP Talkie · Laxmi Kant Tiwari · kgptalkie.com

**Today:** get an **object** back instead of a paragraph. Describe the shape you want once, constrain fields to valid answers, nest shapes, and see exactly how extraction fails, because it fails quietly.

In [2]:
import time

from dotenv import load_dotenv
from langchain_openrouter import ChatOpenRouter

load_dotenv()

model = ChatOpenRouter(model="nvidia/nemotron-3.5-lightning:free", temperature=0)

## 1. Why a paragraph is useless to a program

In [4]:
review = "Arrived two days late but the quality is excellent. Would buy again."

model.invoke("How positive is this review? " + review).text

'This is a **strongly positive review**. \n\nWhile there\'s a minor negative point (arrived two days late), the customer immediately pivots to highlight **excellent quality** and closes with **"Would buy again"** — which is one of the clearest signals of satisfaction a buyer can give. The positive elements (quality, repurchase intent) far outweigh the delivery delay, making it a favorable, confidence-boosting review for any product or seller. \n\nIf you\'re rating it on a positivity scale, I\'d put it around **8.5/10** or "highly positive."'

Fine for a human, useless for a program. You can't store it in a column, count it, or sort by it, and parsing English is what you were trying to avoid.

## 2. Describe the shape with a Pydantic model

A small class. No magic.

In [5]:
from pydantic import BaseModel, Field

class ReviewScore(BaseModel):
    rating: int = Field(description="1 is terrible, 5 is excellent")
    positive: bool = Field(description="true if the customer would buy again")
    reason: str = Field(description="one short sentence")

The `description` text is not a comment. **The model reads it**, sent as part of the request. Vague descriptions produce vague fields.

## 3. Get an object back with with_structured_output

In [7]:
model

ChatOpenRouter(metadata={'lc_versions': {'langchain-core': '1.5.3', 'langchain': '1.3.14', 'langchain-openrouter': '0.2.7'}}, client=<openrouter.sdk.OpenRouter object at 0x00000186811034D0>, openrouter_api_key=SecretStr('**********'), app_url='https://docs.langchain.com', app_title='LangChain', model_name='nvidia/nemotron-3.5-lightning:free', temperature=0.0, model_kwargs={})

In [10]:
reader = model.with_structured_output(ReviewScore)

result = reader.invoke(review)

result

ReviewScore(rating=4, positive=True, reason='Quality excellent, arrived two days late')

In [ ]:
result = reader.invoke('i am not gonna buy your product')

result

In [9]:
reader

_ChatModelBinding(bound=ChatOpenRouter(metadata={'lc_versions': {'langchain-core': '1.5.3', 'langchain': '1.3.14', 'langchain-openrouter': '0.2.7'}}, client=<openrouter.sdk.OpenRouter object at 0x00000186811034D0>, openrouter_api_key=SecretStr('**********'), app_url='https://docs.langchain.com', app_title='LangChain', model_name='nvidia/nemotron-3.5-lightning:free', temperature=0.0, model_kwargs={}), kwargs={'tools': [{'type': 'function', 'function': {'name': 'ReviewScore', 'description': '', 'parameters': {'properties': {'rating': {'description': '1 is terrible, 5 is excellent', 'type': 'integer'}, 'positive': {'description': 'true if the customer would buy again', 'type': 'boolean'}, 'reason': {'description': 'one short sentence', 'type': 'string'}}, 'required': ['rating', 'positive', 'reason'], 'type': 'object'}}}], 'ls_structured_output_format': {'kwargs': {'method': 'function_calling', 'strict': None}, 'schema': {'type': 'function', 'function': {'name': 'ReviewScore', 'description

Not a paragraph. An object.

In [12]:
result.rating

1

In [13]:
result.positive

False

In [14]:
result.model_dump()

{'rating': 1, 'positive': False, 'reason': 'User decided not to purchase.'}

Now it's data: store it, count it, chart it, branch on it. `rating` is declared `int`, so if the model said "four" you'd find out here, not three screens later.

## 4. Limit a field to fixed answers with Literal

`int` still allows 47. When there is a fixed set of valid answers, say so.

In [20]:
from typing import Literal

class Ticket(BaseModel):
    category: Literal["delivery", "billing", "product", "account", "others"] = Field(
        description="which team should handle this"
    )
    urgency: Literal["low", "medium", "high"]
    summary: str = Field(description="the customer problem in under 12 words")

In [21]:
time.sleep(4)

router = model.with_structured_output(Ticket)

# router.invoke("I was charged twice for the same order and nobody has replied for a week.")
router.invoke("How are you?")

Ticket(category='others', urgency='low', summary='User greeting')

`category` can't come back as "money" or anything else needing normalisation, it's one of four strings, enforced by Pydantic.

The single most useful trick here: a `Literal` turns a model into something you can safely write an `if` against.

## 5. Optional fields come back empty even when the answer is there

Not every review mentions a delivery date. The obvious move: make the field optional, so the model can say nothing instead of inventing a number.

In [22]:
class Feedback(BaseModel):
    """Extract delivery details from a customer review."""
    product: str = Field(description="the product mentioned")
    days_late: int | None = Field(
        default=None,
        description="how many days late, or null if the review does not say",
    )

In [23]:
time.sleep(4)

optional = model.with_structured_output(Feedback)

optional.invoke("The blue mug is lovely.")

Feedback(product='blue mug', days_late=None)

`None`, correctly, the model invented nothing. Now a review that plainly does mention a date:

In [25]:
# time.sleep(4)

optional.invoke("The blue mug arrived four days late but it is lovely.")

Feedback(product='blue mug', days_late=4)

The text says **four days late**, the field came back empty anyway. Not a one-off: an optional field doesn't just permit skipping, it lowers the model's effort to even look. "You may leave this out" is a strong hint.

### The fix: a required field plus a bool for absent

In [26]:
class BetterFeedback(BaseModel):
    """Extract delivery details from a customer review."""
    product: str = Field(description="the product mentioned")
    mentions_lateness: bool = Field(description="true if the review says the delivery was late")
    days_late: int = Field(description="number of days late, or 0 if mentions_lateness is false")

In [27]:
# time.sleep(4)

required = model.with_structured_output(BetterFeedback)

required.invoke("The blue mug arrived four days late but it is lovely.")

BetterFeedback(product='blue mug', mentions_lateness=True, days_late=4)

In [28]:
# time.sleep(4)

required.invoke("The blue mug is lovely.")

BetterFeedback(product='blue mug', mentions_lateness=False, days_late=0)

Both correct now: the number is found when present, and the boolean carries "not mentioned" instead of overloading `None`.

> If a value matters, make the field **required** and give the model a defined way to express *absent*. Reach for `| None` only when the field genuinely doesn't matter much.

Test schemas on inputs where you already know the answer, nothing raises on a wrong `None`.

`BetterFeedback` also has a **docstring**: sent to the model as the schema description, it measurably improves extraction.

## 6. Nested models and lists of models

Schemas nest, and lists work too.

In [29]:
class Issue(BaseModel):
    kind: Literal["late", "damaged", "wrong item", "billing", "other"]
    detail: str = Field(description="a few words")

class Complaint(BaseModel):
    customer_mood: Literal["calm", "annoyed", "angry"]
    issues: list[Issue] = Field(description="every distinct problem mentioned")
    wants_refund: bool

In [30]:
# time.sleep(4)

analyst = model.with_structured_output(Complaint)

complaint = analyst.invoke("""I ordered a kettle and you sent a toaster. It also
arrived a week late and the box was soaked. I have been charged twice. I want
my money back.""")

complaint

Complaint(customer_mood='annoyed', issues=[Issue(kind='wrong item', detail='received toaster instead of kettle'), Issue(kind='late', detail='arrived a week late'), Issue(kind='damaged', detail='box was soaked'), Issue(kind='billing', detail='charged twice')], wants_refund=True)

In [31]:
for issue in complaint.issues:
    print(issue.kind + ": " + issue.detail)

wrong item: received toaster instead of kettle
late: arrived a week late
damaged: box was soaked
billing: charged twice


One paragraph of English, one call, and a structure you can walk with a `for`
loop.